In [ ]:
import laya
from laya import Router

In [ ]:
SOM = laya.load (r"C:\Users\Admin\Desktop\models\System One Models\Laya", device = "cuda")

In [ ]:
state = {
    "text": "Show me the values from the column containing the audio times and calculate the average."
}


questions = {
    "OPERATION": {
        "type": "choice",
        "instructions": "Identifies the statistical operation requested by the user."
                        "You choose only one of the available trades.",
        "criteria": {
            "RETURN": "Returns ALL values in a column.",
            "COUNT": "Returns the NUMBER of rows in a column.",
        }
    },
    "COLUMNS": {
        "type": "choice",
        "instructions": "Identifies the column on which the operation should be performed."
                        "Chooses only the column explicitly related to the request.",
        "criteria": {
            "AUDIO_TIME": "Tempo em segundos, de cada áudio.",
            "TEMPO_PRÉ_PROCESSAMENTO": "Tempo em segundos, demorado para realizar o pré processamento do áudio.",
            "TRANSCRIÇÃO": "Transcrição realizada pelo modelo de Automatic Speech Recognition.",
            "TEMPO_PROCESSAMENTO_MODELO_ASR": "Tempo em segundos, que o modelo ASR demorou para realizar o processamento do Áudio, esta parte refere-se ao Encoding do áudio.",
            "TEMPO_INFERÊNCIA_MODELO_ASR": "Tempo em segundos, que o modelo ASR demorou para realizar a inferência após processamento do Áudio, esta parte refere-se ao Decoding do áudio.",
            "LATÊNCIA": "Tempo em segundos, que o modelo ASR demorou a realizar a transcrição do áudio.",
            "TOKENS_PER_SECOND_DECODE": "Throughput do modelo ASR na fase de Inferência, em tokens/s.",
            "HARDWARE_LLM": "Hardware utilizado para realizar a inferência do modelo de Linguagem.",
            "MODELO_LLM": "Modelo de Linguagem utilizado para realizar a Auditoria ás transcrições de áudio.",
            "AUDITORIA_LLM": "Auditoria realizada pelo modelo de Linguagem.",
            "NÚMERO_DE_TOKENS_PROCESSADOS": "Número de tokens processados pelo modelo de Linguagem.",
            "TEMPO_PREFILL_LLM": "Tempo demorado em segundos pelo modelo de Linguagem a realizar a fase de Prefill.",
            "TOKENS_PER_SECOND_PREFILL_LLM": "Throughput realizado pelo modelo de Linguagem ao realizar a fase de Prefill, em tokens/s.",
            "TEMPO_DECODE_LLM": "Tempo demorado em segundos pelo modelo de Linguagem a realizar a fase de Decode.",
            "TOKENS_PER_SECOND_DECODE_LLM": "Throughput realizado pelo modelo de Linguagem ao realizar a fase de Decode, em tokens/s.",
            "LATÊNCIA_LLM": "Tempo total, em segundos, da fase de Inferência do modelo de Linguagem."
        }
    },
}

# Inferência
result = SOM.predict(state, questions)

#print(result)
print(result["answers"]["OPERATION"]["choice"])
print (result["answers"]["OPERATION"]["confidence"])

print(result["answers"]["COLUMNS"]["choice"])
print(result["answers"]["COLUMNS"]["confidence"])


<hr>

<h1> Benchmarking </h1>

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import laya
import yaml
import json

import outlines
from pydantic import BaseModel


c:\Users\Admin\Desktop\ip\Automatic Speech Recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class LOAD_MODEL:

    """
    Load Model
    """
    def __init__ (self):

        PATH = r"C:\Users\Admin\Desktop\models\Language Models\CPU\Mistral 7B Q4 BnB"

        self.device = "cuda" if torch.cuda.is_available () else "cpu"

        self.TOKENIZER = AutoTokenizer.from_pretrained (PATH)
        self.MODEL = AutoModelForCausalLM.from_pretrained (PATH, device_map = self.device, dtype = torch.float16)

#---------------------#

class EVAL:

    def __init__ (self, MODEL, TOKENIZER, DEVICE):

        self.MODEL = MODEL
        self.TOKENIZER = TOKENIZER
        self.DEVICE = DEVICE

        self.MODEL = outlines.from_transformers (self.MODEL, self.TOKENIZER)

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Backend\AssobioChat\SystemPrompts\SYSTEM_PROMPT_SMLAYER.md", "r", encoding = "utf-8") as f:
            self.SYSTEM_PROMPT = f.read ()

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Backend\AssobioChat\SystemPrompts\SemanticModel.yaml", "r", encoding = "utf-8") as f:
            self.SEMANTIC_MODEL = yaml.safe_load (f) 

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Eval\dataset_sml.json", "r", encoding = "utf-8") as f:
            self.DATASET = json.load (f)

    def STRUCTURED_OUTPUT (self):

        class Format_Constraint (BaseModel):
            operation: str
            columns: str

        PRECISION = []
        for i in range (len(self.DATASET)):

            PROMPT = self.DATASET[i]["prompt"]

            MENSAGENS = [
                {"role": "system", "content": f"{self.SYSTEM_PROMPT}\n" f"{self.SEMANTIC_MODEL}"},
                {"role": "user", "content": PROMPT}
            ]

            MENSAGENS = self.TOKENIZER.apply_chat_template (MENSAGENS, tokenize = False, add_generation_prompt = True)
            print (MENSAGENS)

            SEMANTIC_QUERY = self.MODEL (MENSAGENS, output_type = Format_Constraint, max_new_tokens = 50)
            SEMANTIC_QUERY = json.loads (SEMANTIC_QUERY)

            ##Eval

            if SEMANTIC_QUERY["operation"] == self.DATASET[i]["expected_output"]["operation"] and SEMANTIC_QUERY["columns"] == self.DATASET[i]["expected_output"]["columns"]:
                PRECISION.append (1)

            else:
                PRECISION.append (0)

        return list(PRECISION)


if __name__ == "__main__":

    MODELO = LOAD_MODEL ()
    #print (dir(MODELO))
    BENCH = EVAL (MODELO.MODEL, MODELO.TOKENIZER, MODELO.device)
    PRECISION = BENCH.STRUCTURED_OUTPUT ()

    print (PRECISION)

In [ ]:
"""
with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Eval\dataset_sml.json", "r", encoding = "utf-8") as f:
    DATASET = json.load (f)

print (type(DATASET))
print (DATASET)
print (len (DATASET))

for x in range (len (DATASET)):

    print (DATASET[x]["prompt"])
"""